# PyTorch Pruning Benchmarks

Portfolio source extract. See the repository README and PROVENANCE.md for methods, data requirements, original evidence and publication changes. Run from the notebooks/ directory.

In [ ]:
import copy, gc, io, random, time
from pathlib import Path

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.linear_model import lasso_path
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet18

SEED, BATCH_SIZE = 42, 128
TARGET_SPARSITY = 0.80
FINETUNE_EPOCHS, SCRATCH_EPOCHS = 30, 120
DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
DATA_DIR, CHECKPOINT, OUT = Path("data"), Path("resnet18.pt"), Path("results_clean")
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2471, 0.2435, 0.2616)
if not CHECKPOINT.is_file():
    gdown.download(id="17fmN8eQdLpq2jIMQ_X0IXDPXfI9oVWgq", output=str(CHECKPOINT), quiet=False)
OUT.mkdir(exist_ok=True)

def seed_all(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elif DEVICE == "mps":
        torch.mps.synchronize()

seed_all()
print(f"Device: {DEVICE} | target sparsity: {TARGET_SPARSITY:.0%}")

In [ ]:
def new_model(checkpoint=None):
    model = resnet18(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
    nn.init.kaiming_normal_(model.conv1.weight, mode="fan_out", nonlinearity="relu")
    if checkpoint is not None:
        if not Path(checkpoint).is_file():
            raise FileNotFoundError(f"Missing official checkpoint: {checkpoint}")
        state = torch.load(checkpoint, map_location="cpu", weights_only=True)
        model.load_state_dict(state, strict=True)
    return model

model_check = new_model()
assert sum(p.numel() for p in model_check.parameters()) == 11_173_962
assert model_check(torch.randn(2, 3, 32, 32)).shape == (2, 10)
BASE_ELIGIBLE = sum(layer.weight.numel() for layer in model_check.modules()
                    if isinstance(layer, (nn.Conv2d, nn.Linear)))
print("Architecture check passed: 11,173,962 parameters")

In [ ]:
def make_loaders():
    plain = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    augment = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                  transforms.RandomHorizontalFlip(), *plain.transforms])
    train_aug = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=augment)
    train_plain = datasets.CIFAR10(DATA_DIR, train=True, download=False, transform=plain)
    test = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=plain)

    rng, labels = np.random.default_rng(SEED), np.asarray(train_plain.targets)
    groups = [rng.permutation(np.flatnonzero(labels == label)) for label in range(10)]
    val_ids = np.concatenate([ids[:500] for ids in groups]).tolist()
    train_ids = np.concatenate([ids[500:] for ids in groups]).tolist()
    calibration_ids = np.concatenate([ids[500:520] for ids in groups]).tolist()
    rng.shuffle(train_ids)

    def loader(dataset, batch_size=BATCH_SIZE, shuffle=False):
        return DataLoader(dataset, batch_size, shuffle=shuffle, num_workers=2,
                          pin_memory=DEVICE == "cuda")

    return {"train": loader(Subset(train_aug, train_ids), shuffle=True),
            "train_eval": loader(train_plain), "val": loader(Subset(train_plain, val_ids)),
            "test": loader(test),
            "calibration": loader(Subset(train_plain, calibration_ids), 20)}

@torch.no_grad()
def evaluate(model, loader):
    model.to(DEVICE).eval()
    confusion = torch.zeros(10, 10, dtype=torch.long, device=DEVICE)
    loss_sum = top5 = count = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss_sum += F.cross_entropy(logits, y, reduction="sum").item()
        top5 += (logits.topk(5, 1).indices == y[:, None]).any(1).sum().item()
        confusion += torch.bincount(y * 10 + logits.argmax(1), minlength=100).reshape(10, 10)
        count += len(y)
    tp = confusion.diag().float()
    f1 = 2 * tp / (confusion.sum(0) + confusion.sum(1)).clamp_min(1)
    return {"loss": loss_sum / count, "top1": 100 * tp.sum().item() / count,
            "top5": 100 * top5 / count, "macro_f1": f1.mean().item()}

def eligible(model):
    return {name: layer for name, layer in model.named_modules()
            if isinstance(layer, (nn.Conv2d, nn.Linear))}

def full_masks(model):
    return {name: torch.ones_like(layer.weight, dtype=torch.bool)
            for name, layer in eligible(model).items()}

@torch.no_grad()
def apply_masks(model, masks, optimizer=None):
    for name, layer in eligible(model).items():
        if name not in masks:
            continue
        mask = masks[name].to(layer.weight.device)
        layer.weight.mul_(mask)
        if optimizer:
            for value in optimizer.state.get(layer.weight, {}).values():
                if torch.is_tensor(value) and value.shape == mask.shape:
                    value.mul_(mask)

def train_epoch(model, loader, optimizer, masks=None):
    model.train()
    masks = masks or {}
    loss_sum = correct = count = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        for name, layer in eligible(model).items():
            if name in masks:
                layer.weight.grad.mul_(masks[name].to(DEVICE))
        optimizer.step()
        apply_masks(model, masks, optimizer)
        loss_sum += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        count += len(y)
    return {"loss": loss_sum / count, "top1": 100 * correct / count}

def history_row(epoch, train, val):
    return {"epoch": epoch + 1, "train_loss": train["loss"], "train_top1": train["top1"],
            "val_loss": val["loss"], "val_top1": val["top1"]}

def fit(model, name, epochs, lr, masks=None):
    model.to(DEVICE)
    optimizer = torch.optim.SGD(model.parameters(), lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    history = []
    for epoch in range(epochs):
        seed_all(SEED + epoch)
        train = train_epoch(model, loaders["train"], optimizer, masks)
        history.append(history_row(epoch, train, evaluate(model, loaders["val"])))
        scheduler.step()
        print(name, history[-1])
    return history

def plot_history(history, title):
    frame = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for axis, metric, label in zip(axes, ("loss", "top1"), ("Loss", "Top-1 (%)")):
        axis.plot(frame.epoch, frame[f"train_{metric}"], label="train")
        axis.plot(frame.epoch, frame[f"val_{metric}"], label="validation")
        axis.set(xlabel="Epoch", ylabel=label)
        axis.grid(alpha=0.2)
        axis.legend()
    fig.suptitle(title)
    plt.show()


In [ ]:
class COOConv(nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.weight = layer.weight.detach().cpu().to_sparse_coo().coalesce()
        self.register_buffer("bias", None if layer.bias is None else layer.bias.detach().clone())
        self.stride, self.padding = layer.stride, layer.padding
        self.dilation, self.groups = layer.dilation, layer.groups

    def forward(self, x):
        return F.conv2d(x, self.weight.to_dense().to(x.device), self.bias, self.stride,
                        self.padding, self.dilation, self.groups)

class COOLinear(nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.weight = layer.weight.detach().cpu().to_sparse_coo().coalesce()
        self.register_buffer("bias", None if layer.bias is None else layer.bias.detach().clone())

    def forward(self, x):
        result = (torch.sparse.mm(self.weight.to(x.device), x.T).T
                  if x.device.type in {"cpu", "cuda"}
                  else F.linear(x, self.weight.to_dense().to(x.device)))
        return result if self.bias is None else result + self.bias

def coo_model(model):
    if isinstance(model, nn.Conv2d):
        return COOConv(model)
    if isinstance(model, nn.Linear):
        return COOLinear(model)
    result = copy.deepcopy(model).eval()
    for name, child in list(result.named_children()):
        setattr(result, name, coo_model(child))
    return result

def save_coo(model, masks, path):
    state = {key: value.detach().cpu() for key, value in model.state_dict().items()}
    layers = {}
    for name, layer in eligible(model).items():
        sparse = state.pop(f"{name}.weight").to_sparse_coo().coalesce()
        layers[name] = {"indices": sparse.indices(), "values": sparse.values(),
                        "shape": tuple(sparse.shape), "mask": masks[name].cpu()}
    torch.save({"layers": layers, "other_state": state}, path)
    return path.stat().st_size / 1e6

def serialized_mb(model):
    stream = io.BytesIO()
    torch.save(model.state_dict(), stream)
    return stream.tell() / 1e6

@torch.inference_mode()
def count_macs(model, x):
    total, handles = 0, []
    def hook(layer, inputs, output):
        nonlocal total
        if isinstance(layer, (nn.Conv2d, COOConv)):
            total += output.numel() * np.prod(layer.weight.shape[1:])
        elif isinstance(layer, COOLinear):
            total += inputs[0].shape[0] * layer.weight._nnz()
        else:
            total += output.numel() * layer.in_features
    for layer in model.modules():
        if isinstance(layer, (nn.Conv2d, nn.Linear, COOConv, COOLinear)):
            handles.append(layer.register_forward_hook(hook))
    model(x)
    for handle in handles:
        handle.remove()
    return int(total // len(x))

class EnergyMeter:
    def __init__(self):
        self.cpu, self.nvml, self.handle = [], None, None
        for path in Path("/sys/class/powercap").glob("intel-rapl:*"):
            try:
                if path.name.count(":") == 1:
                    self.cpu.append((path / "energy_uj",
                                     int((path / "max_energy_range_uj").read_text())))
            except (OSError, ValueError):
                pass
        if DEVICE == "cuda":
            try:
                import pynvml
                pynvml.nvmlInit()
                self.nvml = pynvml
                self.handle = pynvml.nvmlDeviceGetHandleByIndex(torch.cuda.current_device())
                pynvml.nvmlDeviceGetTotalEnergyConsumption(self.handle)
            except Exception:
                self.nvml = self.handle = None

    def read(self):
        cpu = [int(path.read_text()) for path, _ in self.cpu] if self.cpu else None
        gpu = self.nvml.nvmlDeviceGetTotalEnergyConsumption(self.handle) if self.handle else None
        return cpu, gpu

    def delta(self, start, end):
        cpu = None if start[0] is None else sum(
            (b - a) % maximum for a, b, (_, maximum) in zip(start[0], end[0], self.cpu)
        ) / 1e6
        gpu = None if start[1] is None else (end[1] - start[1]) / 1000
        return cpu, gpu

@torch.inference_mode()
def profile(model):
    model.to(DEVICE).eval()
    x = next(iter(loaders["test"]))[0].to(DEVICE)
    macs = count_macs(model, x[:1])
    for _ in range(10):
        model(x)
    sync()
    meter, times, memory = EnergyMeter(), [], []
    before = meter.read()
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for _ in range(30):
        sync()
        started = time.perf_counter()
        model(x)
        sync()
        times.append(1000 * (time.perf_counter() - started))
        if DEVICE == "cuda":
            memory.append(torch.cuda.memory_allocated() / 1e6)
        elif DEVICE == "mps":
            memory.append(torch.mps.current_allocated_memory() / 1e6)
    cpu_j, gpu_j = meter.delta(before, meter.read())
    peak = (torch.cuda.max_memory_allocated() / 1e6 if DEVICE == "cuda"
            else max(memory) if memory else None)
    return {"macs": macs, "flops": 2 * macs, "latency_ms_batch": np.mean(times),
            "peak_gpu_mb": peak, "average_gpu_mb": np.mean(memory) if memory else None,
            "cpu_energy_j": cpu_j, "gpu_energy_j": gpu_j}

RESULTS = []

def measure(name, model, representation="dense", masks=None, sparse_mb=None, reference=None):
    reference = reference or model
    train, test = evaluate(model, loaders["train_eval"]), evaluate(model, loaders["test"])
    if masks:
        total = sum(mask.numel() for mask in masks.values())
        model_sparsity = 1 - sum(int(mask.sum()) for mask in masks.values()) / total
    else:
        weights = sum(layer.weight.numel() for layer in eligible(reference).values())
        model_sparsity = 1 - weights / BASE_ELIGIBLE
    row = {"name": name, "representation": representation, "sparsity": model_sparsity,
           "parameters": sum(p.numel() for p in reference.parameters()),
           "dense_mb": serialized_mb(reference), "sparse_mb": sparse_mb,
           **{f"train_{key}": value for key, value in train.items()},
           **{f"test_{key}": value for key, value in test.items()}, **profile(model)}
    RESULTS.append(row)
    display(pd.DataFrame([row]))
    model.cpu()
    gc.collect()
    return row


In [ ]:
loaders = make_loaders()
baseline_model = new_model(CHECKPOINT)
baseline_result = measure("baseline", baseline_model)


In [ ]:
LEVELS = [0.0, 0.1, 0.2, 0.5, 0.7, 0.8, 0.9]

def sparsity(masks):
    total = sum(mask.numel() for mask in masks.values())
    return 1 - sum(int(mask.sum()) for mask in masks.values()) / total

def magnitude_scores(model):
    return {name: layer.weight.detach().abs() for name, layer in eligible(model).items()}

def global_masks(scores, previous, target, largest=False):
    names, sizes = list(previous), [mask.numel() for mask in previous.values()]
    mask = torch.cat([previous[name].flatten() for name in names]).clone()
    score = torch.cat([scores[name].flatten() for name in names])
    remove = round(target * mask.numel()) - (mask.numel() - int(mask.sum()))
    if remove:
        active = mask.nonzero().flatten()
        mask[active[torch.topk(score[active], remove, largest=largest).indices]] = False
    return {name: part.reshape_as(previous[name])
            for name, part in zip(names, torch.split(mask, sizes))}

def sensitivity_analysis(model):
    rows = []
    model.to(DEVICE).eval()
    for name, layer in eligible(model).items():
        original = layer.weight.detach().clone()
        for level in LEVELS:
            mask = torch.ones_like(original, dtype=torch.bool).flatten()
            count = round(level * mask.numel())
            if count:
                mask[original.abs().flatten().topk(count, largest=False).indices] = False
            with torch.no_grad():
                layer.weight.copy_(original * mask.reshape_as(original))
            row = {"layer": name, "sparsity": level,
                   "val_top1": evaluate(model, loaders["val"])["top1"]}
            rows.append(row)
            print("sensitivity", row)
        with torch.no_grad():
            layer.weight.copy_(original)
    return pd.DataFrame(rows)

def local_masks(model, curves, target):
    layers = eligible(model)
    sizes = np.array([layer.weight.numel() for layer in layers.values()])
    damage = []
    for name in layers:
        curve = curves[curves.layer == name].sort_values("sparsity")
        damage.append(1 + np.maximum(0, curve.val_top1.iloc[0] - curve.val_top1).mean())
    damage = np.asarray(damage)
    goal, low, high = round(target * sizes.sum()), 0.0, 2 * damage.max()
    for _ in range(60):
        scale = (low + high) / 2
        if np.sum(np.minimum(scale / damage, 0.90) * sizes) < goal:
            low = scale
        else:
            high = scale
    exact = np.minimum(high / damage, 0.90) * sizes
    counts, limits = np.floor(exact).astype(int), np.floor(0.90 * sizes).astype(int)
    for index in np.argsort(exact - counts)[::-1]:
        if counts.sum() == goal:
            break
        if counts[index] < limits[index]:
            counts[index] += 1
    assert counts.sum() == goal
    masks = full_masks(model)
    for (name, layer), count in zip(layers.items(), counts):
        if count:
            ids = layer.weight.detach().abs().flatten().topk(int(count), largest=False).indices
            masks[name].flatten()[ids] = False
    return masks

def mask_table(model, masks):
    rows = []
    for name, layer in eligible(model).items():
        mask = masks[name].to(layer.weight.device)
        masked = 1 - mask.float().mean().item()
        actual = 1 - torch.count_nonzero(layer.weight).item() / layer.weight.numel()
        assert torch.all(layer.weight[~mask] == 0)
        rows.append({"layer": name, "mask_sparsity": masked,
                     "weight_sparsity": actual, "remaining_ratio": 1 - masked,
                     "collapsed": not bool(mask.any())})
    return pd.DataFrame(rows)

In [ ]:
sensitivity_df = sensitivity_analysis(new_model(CHECKPOINT))

fig, ax = plt.subplots(figsize=(12, 6))
for name, rows in sensitivity_df.groupby("layer"):
    ax.plot(rows.sparsity * 100, rows.val_top1, marker=".", label=name)
ax.set(xlabel="Layer sparsity (%)", ylabel="Validation Top-1 (%)")
ax.grid(alpha=0.2)
ax.legend(ncol=3, fontsize=7)
plt.show()

template = new_model(CHECKPOINT).to(DEVICE)
local = local_masks(template, sensitivity_df, TARGET_SPARSITY)
sync()
started = time.perf_counter()
magnitude = magnitude_scores(template)
sync()
task1_criterion_seconds = time.perf_counter() - started
global_mask = global_masks(magnitude, full_masks(template), TARGET_SPARSITY)
assert all(mask.any() for mask in global_mask.values()), "Global pruning collapsed a layer"
task1_before, task1_final, task1_layers = {}, {}, {}

for name, masks in (("local", local), ("global", global_mask)):
    model = new_model(CHECKPOINT)
    apply_masks(model, masks)
    task1_before[name] = evaluate(model, loaders["test"])["top1"]
    history = fit(model, name, FINETUNE_EPOCHS, 0.01, masks)
    plot_history(history, f"Task 1 - {name}")
    task1_layers[name] = mask_table(model, masks)
    display(task1_layers[name])
    task1_final[name] = measure(name, model, "dense_masked", masks)
    sparse_mb = save_coo(model, masks, OUT / f"{name}_coo.pt")
    measure(name, coo_model(model), "coo", masks=masks,
            sparse_mb=sparse_mb, reference=model)

baseline_by_layer = sensitivity_df[sensitivity_df.sparsity == 0].set_index("layer").val_top1
sensitive = sensitivity_df[sensitivity_df.sparsity == 0.8].copy()
sensitive["drop_pp"] = sensitive.layer.map(baseline_by_layer) - sensitive.val_top1
print(f"Magnitude criterion time: {task1_criterion_seconds:.6f} s")
print("Most sensitive layers at 80%:")
display(sensitive.nlargest(5, "drop_pp")[["layer", "drop_pp"]])
print("Accuracy recovered:", {name: task1_final[name]["test_top1"] - task1_before[name]
                              for name in ("local", "global")})
print("Global sparsity range:", task1_layers["global"].mask_sparsity.min(),
      "to", task1_layers["global"].mask_sparsity.max())


In [ ]:
def grasp_scores(model, masks, temperature=200.0):
    layers = eligible(model)
    weights = [layer.weight for layer in layers.values()]
    batches = [(x.to(DEVICE), y.to(DEVICE)) for x, y in loaders["calibration"]]
    samples = sum(len(y) for _, y in batches)
    gradient = [torch.zeros_like(weight) for weight in weights]
    hessian_gradient = [torch.zeros_like(weight) for weight in weights]
    active = [masks[name].to(DEVICE) for name in layers]
    model.eval()
    for x, y in batches:
        loss = F.cross_entropy(model(x) / temperature, y, reduction="sum") / samples
        for total, grad, mask in zip(gradient, torch.autograd.grad(loss, weights), active):
            total.add_(grad.detach() * mask)
    for x, y in batches:
        loss = F.cross_entropy(model(x) / temperature, y, reduction="sum") / samples
        grads = torch.autograd.grad(loss, weights, create_graph=True)
        dot = sum((grad * direction).sum() for grad, direction in zip(grads, gradient))
        for total, product in zip(hessian_gradient, torch.autograd.grad(dot, weights)):
            total.add_(product.detach())
    return {name: -weight.detach() * product
            for name, weight, product in zip(layers, weights, hessian_gradient)}

def iterative_run(method, warm_model, warm_optimizer, warm_scheduler, warm_history):
    model = new_model().to(DEVICE)
    model.load_state_dict(copy.deepcopy(warm_model))
    optimizer = torch.optim.SGD(model.parameters(), 0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, SCRATCH_EPOCHS)
    optimizer.load_state_dict(copy.deepcopy(warm_optimizer))
    scheduler.load_state_dict(copy.deepcopy(warm_scheduler))
    history, masks, events = copy.deepcopy(warm_history), full_masks(model), []
    targets = [0.5 * TARGET_SPARSITY, 0.75 * TARGET_SPARSITY, TARGET_SPARSITY]
    thresholds, deadlines = [20, 40, 60], [5, 40, 80]

    for epoch in range(len(history), SCRATCH_EPOCHS):
        if len(events) < 3:
            stage = len(events)
            if stage == 0 or history[-1]["val_top1"] >= thresholds[stage] or epoch >= deadlines[stage]:
                before = evaluate(model, loaders["test"])["top1"]
                sync()
                started = time.perf_counter()
                scores = grasp_scores(model, masks) if method == "grasp" else magnitude_scores(model)
                sync()
                criterion_seconds = time.perf_counter() - started
                masks = global_masks(scores, masks, targets[stage], largest=method == "grasp")
                apply_masks(model, masks, optimizer)
                layers = mask_table(model, masks)
                events.append({"stage": stage + 1, "epoch": epoch, "test_before": before,
                               "test_after": evaluate(model, loaders["test"])["top1"],
                               "sparsity": sparsity(masks),
                               "criterion_seconds": criterion_seconds,
                               "reason": ("accuracy reached" if history[-1]["val_top1"] >= thresholds[stage]
                                          else "epoch cap"),
                               "collapsed_layers": int(layers.collapsed.sum())})
                display(pd.DataFrame([events[-1]]))

        seed_all(SEED + epoch)
        train = train_epoch(model, loaders["train"], optimizer, masks)
        history.append(history_row(epoch, train, evaluate(model, loaders["val"])))
        scheduler.step()
        print(method, history[-1])

    plot_history(history, f"Task 2 - iterative {method}")
    layers = mask_table(model, masks)
    display(layers)
    dense = measure(f"iterative_{method}", model, "dense_masked", masks)
    sparse_mb = save_coo(model, masks, OUT / f"iterative_{method}_coo.pt")
    measure(f"iterative_{method}", coo_model(model), "coo", masks=masks,
            sparse_mb=sparse_mb, reference=model)
    return dense, pd.DataFrame(events), layers


In [ ]:
seed_all()
warm = new_model().to(DEVICE)
torch.save(warm.state_dict(), OUT / "task2_random_initial.pt")
optimizer = torch.optim.SGD(warm.parameters(), 0.1, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, SCRATCH_EPOCHS)
warm_history = []
while len(warm_history) < 5 and (not warm_history or warm_history[-1]["val_top1"] < 20):
    epoch = len(warm_history)
    train = train_epoch(warm, loaders["train"], optimizer)
    warm_history.append(history_row(epoch, train, evaluate(warm, loaders["val"])))
    scheduler.step()
    print("warm-up", warm_history[-1])

warm_state = {"model": copy.deepcopy(warm.state_dict()),
              "optimizer": copy.deepcopy(optimizer.state_dict()),
              "scheduler": copy.deepcopy(scheduler.state_dict()),
              "history": warm_history}
torch.save(warm_state, OUT / "task2_warmup.pt")
warm.cpu()

task2 = {}
for method in ("magnitude", "grasp"):
    task2[method] = iterative_run(method, warm_state["model"], warm_state["optimizer"],
                                  warm_state["scheduler"], warm_history)
comparison = pd.DataFrame([
    {"method": method, "test_top1": values[0]["test_top1"],
     "collapsed_layers": int(values[2].collapsed.sum()),
     "criterion_seconds": values[1].criterion_seconds.sum()}
    for method, values in task2.items()
])
display(comparison)
scores = comparison.set_index("method").test_top1
print("GraSP minus iterative magnitude Top-1:", scores.grasp - scores.magnitude)


In [ ]:
def blocks_to_prune(model):
    blocks = [(f"layer{stage}.{index}", block) for stage in range(1, 5)
              for index, block in enumerate(getattr(model, f"layer{stage}"))]
    return blocks[1:]

def channel_plan(model, target, local_ratios):
    blocks = blocks_to_prune(model)
    total = sum(layer.weight.numel() for layer in eligible(model).values())
    widths = np.array([block.conv1.out_channels for _, block in blocks])
    costs = np.array([block.conv1.weight[0].numel() + block.conv2.weight[:, 0].numel()
                      for _, block in blocks])
    preference = np.array([max(0.01, local_ratios.get(f"{name}.conv2", target))
                           for name, _ in blocks])
    goal, low, high = round(total * target), 0.0, 1 / preference.min()
    for _ in range(60):
        scale = (low + high) / 2
        removed = np.minimum(widths - 1, widths * preference * scale)
        if np.sum(removed * costs) < goal:
            low = scale
        else:
            high = scale
    removed = np.floor(np.minimum(widths - 1, widths * preference * high)).astype(int)
    while True:
        current = np.sum(removed * costs)
        choices = [(abs(goal - current - costs[i]), i) for i in range(len(removed))
                   if removed[i] < widths[i] - 1]
        if not choices or min(choices)[0] >= abs(goal - current):
            break
        removed[min(choices)[1]] += 1
    return {name: int(width - cut)
            for (name, _), width, cut in zip(blocks, widths, removed)}

@torch.inference_mode()
def collect_pairs(model, conv, max_rows=2048):
    xs, ys = [], []
    per_batch = max(1, int(np.ceil(max_rows / len(loaders["calibration"]))))
    generator = torch.Generator().manual_seed(SEED)

    def hook(layer, inputs, output):
        patches = F.unfold(inputs[0], layer.kernel_size, dilation=layer.dilation,
                           padding=layer.padding, stride=layer.stride)
        patches = patches.transpose(1, 2).reshape(-1, patches.shape[1])
        targets = output.permute(0, 2, 3, 1).reshape(-1, output.shape[1])
        ids = torch.randperm(len(patches), generator=generator)[:per_batch].to(DEVICE)
        xs.append(patches[ids].cpu())
        ys.append(targets[ids].cpu())

    handle = conv.register_forward_hook(hook)
    try:
        model.eval()
        for x, _ in loaders["calibration"]:
            model(x.to(DEVICE))
    finally:
        handle.remove()
    return torch.cat(xs)[:max_rows], torch.cat(ys)[:max_rows]

def select_channels(x, y, weight, keep):
    outputs, channels, kh, kw = weight.shape
    rng = np.random.default_rng(SEED)
    rows = rng.choice(len(x) * outputs, min(4096, len(x) * outputs), replace=False)
    positions, output_ids = rows // outputs, rows % outputs
    patches = x.reshape(-1, channels, kh * kw)
    filters = weight.reshape(outputs, channels, kh * kw)
    z = (patches[positions] * filters[output_ids]).sum(-1).double().numpy()
    target = y[positions, output_ids].double().numpy()
    alpha_max = max(float(np.max(np.abs(z.T @ target))) / len(target), 1e-12)
    alphas = np.geomspace(alpha_max, alpha_max * 1e-5, 80)
    values, coefficients, _ = lasso_path(z, target, alphas=alphas, max_iter=10000)
    counts = np.count_nonzero(np.abs(coefficients) > 1e-10, axis=0)
    index = min(np.flatnonzero(counts >= keep), key=lambda i: (counts[i] - keep, i))
    beta = coefficients[:, index]
    chosen = np.sort(np.argsort(-np.abs(beta), kind="stable")[:keep])
    return torch.tensor(chosen), torch.tensor(beta[chosen], dtype=x.dtype), values[index]

def reconstruct(x, y, ids, beta, kernel):
    kh, kw = kernel
    reduced = x.reshape(len(x), -1, kh * kw)[:, ids]
    design = (reduced * beta[None, :, None]).flatten(1).double()
    solution = torch.linalg.lstsq(design, y.double(), driver="gelsd").solution
    folded = solution.reshape(len(ids), kh * kw, y.shape[1]) * beta.double()[:, None, None]
    filters = folded.permute(2, 0, 1).reshape(y.shape[1], len(ids), kh, kw).float()
    error = F.mse_loss(reduced.flatten(1) @ filters.flatten(1).T, y).item()
    return filters, error

def shrink_block(block, ids, reconstructed):
    ids = ids.to(block.conv1.weight.device)
    first = nn.Conv2d(block.conv1.in_channels, len(ids), block.conv1.kernel_size,
                      block.conv1.stride, block.conv1.padding, bias=False).to(DEVICE)
    second = nn.Conv2d(len(ids), block.conv2.out_channels, block.conv2.kernel_size,
                       block.conv2.stride, block.conv2.padding, bias=False).to(DEVICE)
    bn = nn.BatchNorm2d(len(ids), eps=block.bn1.eps, momentum=block.bn1.momentum).to(DEVICE)
    with torch.no_grad():
        first.weight.copy_(block.conv1.weight[ids])
        second.weight.copy_(reconstructed.to(DEVICE))
        for target, source in ((bn.weight, block.bn1.weight), (bn.bias, block.bn1.bias),
                               (bn.running_mean, block.bn1.running_mean),
                               (bn.running_var, block.bn1.running_var)):
            target.copy_(source[ids])
        bn.num_batches_tracked.copy_(block.bn1.num_batches_tracked)
    block.conv1, block.bn1, block.conv2 = first, bn, second

def structured_prune(model, plan):
    model.to(DEVICE).eval()
    rows = []
    for name, block in blocks_to_prune(model):
        started, old = time.perf_counter(), block.conv2.in_channels
        x, y = collect_pairs(model, block.conv2)
        ids, beta, alpha = select_channels(x, y, block.conv2.weight.detach().cpu(), plan[name])
        reconstructed, error = reconstruct(x, y, ids, beta, block.conv2.kernel_size)
        shrink_block(block, ids, reconstructed)
        row = {"block": name, "original_channels": old, "kept_channels": len(ids),
               "alpha": alpha, "reconstruction_mse": error,
               "seconds": time.perf_counter() - started}
        rows.append(row)
        print("structured", row)
    return model, pd.DataFrame(rows)

In [ ]:
local_ratios = dict(zip(task1_layers["local"].layer,
                        task1_layers["local"].mask_sparsity))
structured = new_model(CHECKPOINT)
plan = channel_plan(structured, TARGET_SPARSITY, local_ratios)
structured, channel_table = structured_prune(structured, plan)
display(channel_table)
structured_before = evaluate(structured, loaders["test"])["top1"]
history = fit(structured, "structured", FINETUNE_EPOCHS, 0.01)
plot_history(history, "Task 3 - structured pruning")
structured_result = measure("structured", structured)

baseline = next(row for row in RESULTS if row["name"] == "baseline")
print("Accuracy recovered:", structured_result["test_top1"] - structured_before)
print("Parameter reduction:", 1 - structured_result["parameters"] / baseline["parameters"])
print("Latency speedup:", baseline["latency_ms_batch"] / structured_result["latency_ms_batch"])
print("Accuracy drop:", baseline["test_top1"] - structured_result["test_top1"])


In [ ]:
final_results = pd.DataFrame(RESULTS)
display(final_results)
dense = final_results[final_results.representation != "coo"].set_index("name")
summary = pd.DataFrame({
    "sparsity": dense.sparsity,
    "test_top1": dense.test_top1,
    "accuracy_drop_pp": dense.loc["baseline", "test_top1"] - dense.test_top1,
    "latency_speedup": dense.loc["baseline", "latency_ms_batch"] / dense.latency_ms_batch,
    "parameter_reduction": 1 - dense.parameters / dense.loc["baseline", "parameters"],
})
display(summary)
